# PayJoy FPD_15 Prediction — Model Experiments

Experimentation notebook for testing model architectures, feature combinations,
and hyperparameter configurations. Primary metric: **AUC-ROC**.

In [ ]:
import numpy as np
np.random.seed(42)  # Pin global seed for reproducible train/test splits and model init
import pandas as pd
from itertools import product  # Used in Module 4 to build the NN hyperparameter grid

import matplotlib
matplotlib.use('Agg')  # Non-blocking backend for headless/automation runs
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve  # AUC-ROC is the competition metric

import xgboost as xgb  # Gradient-boosted trees (Module 2)
import lightgbm as lgb  # Gradient-boosted trees with native categoricals (Module 2B)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset  # PyTorch MLP (Module 3)

from tqdm import tqdm  # Progress bars for the NN grid search loop

import gc
import warnings
warnings.filterwarnings('ignore')  # Suppress sklearn/xgboost convergence noise

In [ ]:
# Orders.csv  — one row per phone-finance order (Jan–Dec 2025).
print('[PROGRESS] Loading data...')
#   Jan–Nov rows have a known FPD_15 label → training data.
#   Dec rows have FPD_15 = NaN → these are the prediction targets.
# Payment_History.csv — monthly payment snapshots for pre-Dec orders (~5.7M rows).
#   December orders have NO payment history (the loan just started).
# Test_OrderIDs.csv — the specific Dec order IDs we must submit predictions for.
orders = pd.read_csv('Orders.csv', low_memory=False)
payments = pd.read_csv('Payment_History.csv')
test_ids = pd.read_csv('Test_OrderIDs.csv')

print(f'Orders:   {orders.shape}')
print(f'Payments: {payments.shape}')
print(f'Test IDs: {len(test_ids)}')

## Module 1 — Data Preparation & Feature Selection

Three functions work together here:

1. `_aggregate_payments` — collapses monthly payment snapshots into one row per order (max paid, worst delinquency, etc.).
2. `_engineer_features` — adds financial ratios, time-of-sale signals, and entity-level fraud rates.
3. `prep_data` — orchestrates the above, then handles imputation, one-hot encoding, and optional scaling. This is the single entry point you call from the execution block.

In [ ]:
def _aggregate_payments(payments_df):
    """Aggregate payment history to order-level features.

    Returns a DataFrame keyed on FINANCEORDERID with columns:
      pay_max_cumpaid, pay_max_overdue, pay_mean_overdue, pay_num_snapshots.

    Note: these columns are only populated for pre-December orders;
    December (test) rows will be NaN and get imputed downstream.
    """
    # Each order can span multiple monthly snapshots in Payment_History.
    # We collapse those into a single row of summary statistics per order.
    agg = payments_df.groupby('FINANCEORDERID').agg(
        pay_max_cumpaid=('TOTAL_CUMPAID', 'max'),        # Lifetime total paid
        pay_max_overdue=('DAYSOVERDUE', 'max'),           # Worst delinquency
        pay_mean_overdue=('DAYSOVERDUE', 'mean'),         # Average delinquency
        pay_num_snapshots=('CALENDAR_DATE', 'nunique'),   # Months of history
    ).reset_index()
    return agg


def _engineer_features(df, rate_mask=None):
    """Derive additional columns from raw order data.

    Creates financial ratios, temporal features, and smoothed (Bayesian)
    target-encoded entity/geographic FPD rates.  Smoothing shrinks noisy
    estimates for rare entities toward the global mean.

    Parameters
    ----------
    df : DataFrame
        The full orders table (possibly enriched with payment aggregations).
    rate_mask : pd.Series[bool] or None
        Boolean mask selecting which rows contribute to target encoding.
        Pass None to use all labelled rows (for final submission model).
        Pass a mask excluding the validation month to prevent leakage.
    """
    # -- Financial ratios -------------------------------------------------
    df['down_payment_ratio'] = (
        df['DOWN_PAYMENT_AMOUNT'] / df['PURCHASE_AMOUNT'].replace(0, np.nan)
    )
    df['finance_ratio'] = (
        df['FINANCE_AMOUNT'] / df['PURCHASE_AMOUNT'].replace(0, np.nan)
    )

    # -- Temporal features ------------------------------------------------
    tx = pd.to_datetime(df['TRANSACTIONTIME'], utc=True)
    df['tx_hour'] = tx.dt.hour
    df['tx_dayofweek'] = tx.dt.dayofweek
    df['tx_month'] = tx.dt.month

    # -- days_to_first_due (temporal AUC = 0.5427) ------------------------
    fpd_ts = pd.to_datetime(df['FIRST_PAYMENT_DUE_TIMESTAMP'], utc=True)
    df['days_to_first_due'] = (fpd_ts - tx).dt.total_seconds() / 86400
    print(f'  days_to_first_due: median={df["days_to_first_due"].median():.1f}, '
          f'range=[{df["days_to_first_due"].min():.1f}, {df["days_to_first_due"].max():.1f}]')

    # -- Merchant tenure --------------------------------------------------
    mfsd = pd.to_datetime(df['MERCHANT_FIRST_SALE_DATE'], utc=True)
    df['merchant_tenure_days'] = (tx - mfsd).dt.days

    # -- Smoothed (Bayesian) target encoding ------------------------------
    if rate_mask is None:
        rate_mask = df['FPD_15'].notna()
    global_mean = df.loc[rate_mask, 'FPD_15'].mean()
    m = 50  # smoothing parameter — shrinks rare-entity estimates toward global mean
    print(f'  Smoothed target encoding: global_mean={global_mean:.4f}, smoothing_m={m}')

    for entity in ['MERCHANTID', 'CLERK_ID', 'ADMINID']:
        grp = df.loc[rate_mask].groupby(entity)['FPD_15'].agg(['mean', 'count'])
        grp['smoothed'] = (grp['count'] * grp['mean'] + m * global_mean) / (grp['count'] + m)
        df[f'{entity.lower()}_fpd_rate'] = df[entity].map(grp['smoothed'])
        print(f'    {entity}: {len(grp)} unique, smoothed rate '
              f'[{grp["smoothed"].min():.4f}, {grp["smoothed"].max():.4f}]')

    # -- State smoothed target encoding -----------------------------------
    state_grp = df.loc[rate_mask].groupby('STATE')['FPD_15'].agg(['mean', 'count'])
    state_grp['smoothed'] = (state_grp['count'] * state_grp['mean'] + m * global_mean) / (state_grp['count'] + m)
    df['state_fpd_rate'] = df['STATE'].map(state_grp['smoothed'])
    print(f'    STATE: {len(state_grp)} unique, smoothed rate '
          f'[{state_grp["smoothed"].min():.4f}, {state_grp["smoothed"].max():.4f}]')

    # -- CITY smoothed target encoding (temporal AUC = 0.5746) ------------
    city_grp = df.loc[rate_mask].groupby('CITY')['FPD_15'].agg(['mean', 'count'])
    city_grp['smoothed'] = (city_grp['count'] * city_grp['mean'] + m * global_mean) / (city_grp['count'] + m)
    df['city_fpd_rate'] = df['CITY'].map(city_grp['smoothed'])
    print(f'    CITY: {len(city_grp)} unique, smoothed rate '
          f'[{city_grp["smoothed"].min():.4f}, {city_grp["smoothed"].max():.4f}]')

    # -- MODEL smoothed target encoding (temporal AUC = 0.5478) -----------
    model_grp = df.loc[rate_mask].groupby('MODEL')['FPD_15'].agg(['mean', 'count'])
    model_grp['smoothed'] = (model_grp['count'] * model_grp['mean'] + m * global_mean) / (model_grp['count'] + m)
    df['model_fpd_rate'] = df['MODEL'].map(model_grp['smoothed'])
    print(f'    MODEL: {len(model_grp)} unique, smoothed rate '
          f'[{model_grp["smoothed"].min():.4f}, {model_grp["smoothed"].max():.4f}]')

    # -- Entity order volume counts ---------------------------------------
    for entity in ['MERCHANTID', 'CLERK_ID', 'ADMINID']:
        counts = df.loc[rate_mask].groupby(entity)['FPD_15'].count()
        df[f'{entity.lower()}_order_count'] = df[entity].map(counts)

    return df

In [ ]:
def prep_data(orders_df, payments_df, selected_features, scale_data=True,
              val_month=None):
    """End-to-end data preparation pipeline.

    Orchestrates payment aggregation, feature engineering, imputation,
    one-hot encoding, and optional scaling into a single call.

    Parameters
    ----------
    orders_df : DataFrame
        Full orders table (rows with non-null FPD_15 form the training set).
    payments_df : DataFrame
        Payment history table.
    selected_features : list[str]
        Column names to include as model inputs (raw or engineered).
    scale_data : bool
        If True, fit a StandardScaler on the training split and transform
        all splits.
    val_month : int or None
        When set (e.g. 11), labelled rows whose transaction month equals
        val_month become the validation set, and entity FPD rates are
        computed from the training fold only (months < val_month) to
        prevent target leakage.  When None, all labelled rows are used
        for training and rate computation (for the final submission model).

    Returns
    -------
    When val_month is None:
        X_train, X_test, y_train, scaler
    When val_month is set:
        X_train, X_val, X_test, y_train, y_val, scaler
    """
    df = orders_df.copy()

    # Step 1 — Enrich orders with per-order payment aggregations.
    pay_agg = _aggregate_payments(payments_df)
    df = df.merge(pay_agg, on='FINANCEORDERID', how='left')

    # Step 2 — Determine the rate computation mask BEFORE engineering features.
    labelled = df['FPD_15'].notna()
    if val_month is not None:
        tx_months = pd.to_datetime(df['TRANSACTIONTIME'], utc=True).dt.month
        rate_mask = labelled & (tx_months != val_month)
    else:
        rate_mask = None  # _engineer_features will default to all labelled

    # Step 3 — Build engineered columns (ratios, time features, entity rates).
    df = _engineer_features(df, rate_mask=rate_mask)

    # Step 4 — Split into train / (optional val) / test.
    if val_month is not None:
        tx_months_all = pd.to_datetime(df['TRANSACTIONTIME'], utc=True).dt.month
        train_mask = labelled & (tx_months_all != val_month)
        val_mask = labelled & (tx_months_all == val_month)
    else:
        train_mask = labelled
        val_mask = None

    y_train = df.loc[train_mask, 'FPD_15'].values
    train_feat = df.loc[train_mask, selected_features].copy()
    test_feat = df.loc[~labelled, selected_features].copy()

    if val_mask is not None:
        y_val = df.loc[val_mask, 'FPD_15'].values
        val_feat = df.loc[val_mask, selected_features].copy()

    # Step 5 — Identify column types for type-specific imputation.
    cat_cols = train_feat.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = train_feat.select_dtypes(include='number').columns.tolist()

    # Step 6 — Impute missing values from TRAINING stats only.
    medians = train_feat[num_cols].median()
    train_feat[num_cols] = train_feat[num_cols].fillna(medians)
    test_feat[num_cols] = test_feat[num_cols].fillna(medians)
    if val_mask is not None:
        val_feat[num_cols] = val_feat[num_cols].fillna(medians)

    for col in cat_cols:
        fill = train_feat[col].mode().iloc[0] if not train_feat[col].mode().empty else 'UNKNOWN'
        train_feat[col] = train_feat[col].fillna(fill)
        test_feat[col] = test_feat[col].fillna(fill)
        if val_mask is not None:
            val_feat[col] = val_feat[col].fillna(fill)

    # Step 7 — One-hot encode categoricals, then align column sets.
    all_splits = [train_feat, test_feat]
    if val_mask is not None:
        all_splits.append(val_feat)

    if cat_cols:
        encoded = [pd.get_dummies(s, columns=cat_cols) for s in all_splits]
        ref = encoded[0]
        aligned = [ref] + [
            ref.align(s, join='left', axis=1, fill_value=0)[1] for s in encoded[1:]
        ]
        train_feat, test_feat = aligned[0], aligned[1]
        if val_mask is not None:
            val_feat = aligned[2]

    # Step 8 — Optional z-score scaling.
    scaler = None
    if scale_data:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_feat.values.astype(np.float64))
        X_test = scaler.transform(test_feat.values.astype(np.float64))
        if val_mask is not None:
            X_val = scaler.transform(val_feat.values.astype(np.float64))
    else:
        X_train = train_feat.values.astype(np.float64)
        X_test = test_feat.values.astype(np.float64)
        if val_mask is not None:
            X_val = val_feat.values.astype(np.float64)

    if val_mask is not None:
        return X_train, X_val, X_test, y_train, y_val, scaler
    return X_train, X_test, y_train, scaler

In [ ]:
def prep_data_lgbm(orders_df, payments_df, selected_features, cat_cols,
                   val_month=None):
    """Data preparation for LightGBM — native categoricals, no OHE.

    Same pipeline as prep_data (aggregate → engineer → impute) but preserves
    categorical columns as pandas category dtype instead of one-hot encoding.
    Returns DataFrames so LightGBM can use native categorical handling.

    Returns
    -------
    When val_month is set:
        X_train_df, X_val_df, X_test_df, y_train, y_val, cat_cols, train_months
    When val_month is None:
        X_train_df, X_test_df, y_train, cat_cols
    """
    df = orders_df.copy()
    pay_agg = _aggregate_payments(payments_df)
    df = df.merge(pay_agg, on='FINANCEORDERID', how='left')

    labelled = df['FPD_15'].notna()
    tx_months_all = pd.to_datetime(df['TRANSACTIONTIME'], utc=True).dt.month

    if val_month is not None:
        rate_mask = labelled & (tx_months_all != val_month)
    else:
        rate_mask = None

    print('  Running feature engineering for LightGBM...')
    df = _engineer_features(df, rate_mask=rate_mask)

    if val_month is not None:
        train_mask = labelled & (tx_months_all != val_month)
        val_mask = labelled & (tx_months_all == val_month)
    else:
        train_mask = labelled
        val_mask = None

    y_train = df.loc[train_mask, 'FPD_15'].values
    train_feat = df.loc[train_mask, selected_features].copy()
    test_feat = df.loc[~labelled, selected_features].copy()
    if val_mask is not None:
        y_val = df.loc[val_mask, 'FPD_15'].values
        val_feat = df.loc[val_mask, selected_features].copy()

    # Impute numerics with training medians
    num_cols = [c for c in selected_features if c not in cat_cols]
    medians = train_feat[num_cols].median()
    train_feat[num_cols] = train_feat[num_cols].fillna(medians)
    test_feat[num_cols] = test_feat[num_cols].fillna(medians)
    if val_mask is not None:
        val_feat[num_cols] = val_feat[num_cols].fillna(medians)

    # Convert categoricals to string → fill NaN → category dtype
    for col in cat_cols:
        fill = 'UNKNOWN'
        train_feat[col] = train_feat[col].fillna(fill).astype(str).replace('nan', fill).replace('<NA>', fill)
        test_feat[col] = test_feat[col].fillna(fill).astype(str).replace('nan', fill).replace('<NA>', fill)
        if val_mask is not None:
            val_feat[col] = val_feat[col].fillna(fill).astype(str).replace('nan', fill).replace('<NA>', fill)

        cats = sorted([str(x) for x in train_feat[col].unique()])
        all_cats = pd.CategoricalDtype(categories=cats)
        train_feat[col] = train_feat[col].astype(all_cats)
        test_feat[col] = test_feat[col].astype(all_cats)
        if val_mask is not None:
            val_feat[col] = val_feat[col].astype(all_cats)

    print(f'  LightGBM features: {len(selected_features)} total '
          f'({len(num_cols)} numeric + {len(cat_cols)} categorical)')

    if val_mask is not None:
        train_months = tx_months_all[train_mask].values
        return train_feat, val_feat, test_feat, y_train, y_val, cat_cols, train_months
    return train_feat, test_feat, y_train, cat_cols

## Module 2 — XGBoost Generation

`build_xgb_model(params)` is a thin factory that merges user-specified hyperparameters with sensible defaults and returns a ready-to-fit `XGBClassifier`. It is used by `tune_xgb` in Module 4 but can also be called standalone for quick one-off experiments.

In [ ]:
def build_xgb_model(params):
    """Instantiate an XGBClassifier with the given hyperparameters.

    Provides sensible defaults (binary logistic objective, AUC metric,
    fixed random seed) that can be overridden by anything in `params`.

    Parameters
    ----------
    params : dict
        Any keyword arguments accepted by xgb.XGBClassifier
        (e.g. max_depth, learning_rate, n_estimators, subsample,
        scale_pos_weight).

    Returns
    -------
    xgb.XGBClassifier
    """
    # Baseline config: binary classification scored by AUC, seeded for
    # reproducibility.  User-supplied params override any of these.
    defaults = dict(
        objective='binary:logistic',  # Outputs probabilities via sigmoid
        eval_metric='auc',
        random_state=42,
    )
    defaults.update(params)
    return xgb.XGBClassifier(**defaults)

In [ ]:
def build_lgbm_model(params):
    """Instantiate an LGBMClassifier with sensible defaults.

    LightGBM supports native categorical features — pass categorical
    column names via the fit() method's categorical_feature parameter.

    Parameters
    ----------
    params : dict
        Any keyword arguments accepted by lgb.LGBMClassifier
        (e.g. num_leaves, learning_rate, n_estimators).

    Returns
    -------
    lgb.LGBMClassifier
    """
    defaults = dict(
        objective='binary',
        metric='auc',
        random_state=42,
        verbose=-1,
    )
    defaults.update(params)
    return lgb.LGBMClassifier(**defaults)

## Module 3 — Neural Network Generation (PyTorch)

`FPDNet` is a dynamic MLP whose depth and width are controlled by a `hidden_layers` list (e.g. `[256, 128]` creates a 2-hidden-layer network). Each hidden block applies **Linear → BatchNorm → Activation → Dropout(0.4)**, and the output layer produces a raw logit (use `BCEWithLogitsLoss` for training, `predict_proba()` for inference). `build_nn_model` is the public constructor.

In [ ]:
class FPDNet(nn.Module):
    """Configurable multi-layer perceptron for binary classification.

    Architecture per hidden layer:
      Linear → BatchNorm → Activation → Dropout(0.4)

    The final layer outputs a raw logit.  Use BCEWithLogitsLoss for
    numerically stable training.  Call predict_proba() for probabilities.
    """

    _ACTIVATIONS = {
        'relu': nn.ReLU,
        'tanh': nn.Tanh,
        'leaky_relu': nn.LeakyReLU,
    }

    def __init__(self, input_dim, hidden_layers, activation='relu'):
        super().__init__()
        act_cls = self._ACTIVATIONS[activation]
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), act_cls(), nn.Dropout(0.4)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def predict_proba(self, x):
        """Apply sigmoid to raw logits to get probabilities."""
        return torch.sigmoid(self.forward(x))


def build_nn_model(input_dim, hidden_layers, activation='relu'):
    """Build and return an FPDNet instance.

    Parameters
    ----------
    input_dim : int
        Number of input features.
    hidden_layers : list[int]
        Neurons per hidden layer, e.g. [256, 128] → 2 hidden layers.
    activation : str
        One of 'relu', 'tanh', 'leaky_relu'.

    Returns
    -------
    FPDNet
    """
    return FPDNet(input_dim, hidden_layers, activation)

## Module 4 — Hyperparameter Tuning & Evaluation Engines

Three utilities live here:

| Function | Purpose |
|---|---|
| `plot_roc_curve` | Visualise model discrimination on a validation set. |
| `tune_xgb` | Randomised CV search over XGBoost hyperparameters (delegates to sklearn). |
| `tune_nn` | Manual grid search over NN architectures with a custom PyTorch training loop. |

Both tuning functions return the best model **and** a results DataFrame so you can inspect how each hyperparameter combination performed.

In [ ]:
def plot_roc_curve(y_true, y_prob, label='Model'):
    """Plot a single ROC curve and print the AUC score.

    The dashed diagonal represents a random classifier (AUC = 0.5).
    The further the curve bows toward the top-left, the better the model
    discriminates between FPD and non-FPD orders.

    Parameters
    ----------
    y_true : array-like — ground-truth binary labels
    y_prob : array-like — predicted probabilities for the positive class
    label  : str        — legend label
    """
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, lw=2, label=f'{label} (AUC = {auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)  # Random-guess baseline
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    safe_label = label.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(f'roc_curve_{safe_label}.png', dpi=100)
    plt.close()
    print(f'{label} AUC: {auc:.4f}')

In [ ]:
def tune_xgb(X_train, y_train, param_grid, cv_folds=5, n_iter=20):
    """Tune an XGBoost classifier via RandomizedSearchCV.

    RandomizedSearchCV samples `n_iter` random combinations from
    `param_grid`, evaluates each with `cv_folds`-fold stratified CV
    scored by AUC-ROC, then refits the best config on the full
    X_train before returning it.

    Parameters
    ----------
    X_train    : np.ndarray
    y_train    : np.ndarray
    param_grid : dict — hyperparameter distributions / lists
    cv_folds   : int  — number of cross-validation folds
    n_iter     : int  — random parameter combinations to evaluate
                        (total fits = n_iter × cv_folds)

    Returns
    -------
    best_model : xgb.XGBClassifier (refit on full X_train)
    results_df : DataFrame — ranked CV results
    """
    base = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        random_state=42,
    )
    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring='roc_auc',      # Competition metric
        cv=cv_folds,             # Stratified by default for classifiers
        random_state=42,
        n_jobs=2,                # limit parallelism to reduce memory/CPU load
        verbose=1,
    )
    search.fit(X_train, y_train)

    # Extract a tidy leaderboard of tested configurations
    results_df = (
        pd.DataFrame(search.cv_results_)
        .sort_values('rank_test_score')
        [['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    )
    print(f'Best CV AUC: {search.best_score_:.4f}')
    print(f'Best params: {search.best_params_}')
    return search.best_estimator_, results_df

In [ ]:
def tune_lgbm(X_train_df, y_train, train_months, param_grid, cat_cols=None,
              n_iter=20):
    """Tune LightGBM via expanding-window temporal CV with random search.

    4-fold expanding-window temporal CV within the training set:
      Fold 1: Train months 1-6, Val month 7
      Fold 2: Train months 1-7, Val month 8
      Fold 3: Train months 1-8, Val month 9
      Fold 4: Train months 1-9, Val month 10

    Parameters
    ----------
    X_train_df   : DataFrame — training features (category dtype preserved)
    y_train      : np.ndarray — training labels
    train_months : np.ndarray — month number for each training row
    param_grid   : dict — hyperparameter lists to sample from
    cat_cols     : list[str] — categorical column names for LightGBM
    n_iter       : int — random configurations to evaluate

    Returns
    -------
    best_model : lgb.LGBMClassifier (refit on full training set)
    results_df : DataFrame — ranked CV results
    """
    import random as _random
    _random.seed(42)

    combos = [{k: _random.choice(v) for k, v in param_grid.items()}
              for _ in range(n_iter)]

    folds = [(6, 7), (7, 8), (8, 9), (9, 10)]
    print(f'LightGBM tuning: {n_iter} random configs × {len(folds)} expanding-window folds')
    print(f'Categorical features: {cat_cols}\n')

    records = []
    best_mean_auc = -1
    best_params = None

    for i, params in enumerate(combos):
        fold_aucs = []
        for max_train_month, val_month in folds:
            tr_idx = np.where(train_months <= max_train_month)[0]
            vl_idx = np.where(train_months == val_month)[0]

            model = build_lgbm_model(params)
            model.fit(X_train_df.iloc[tr_idx], y_train[tr_idx],
                      categorical_feature=cat_cols)
            probs = model.predict_proba(X_train_df.iloc[vl_idx])[:, 1]
            fold_aucs.append(roc_auc_score(y_train[vl_idx], probs))

        mean_auc = np.mean(fold_aucs)
        records.append({**params, 'mean_cv_auc': mean_auc,
                        'fold_aucs': [round(a, 4) for a in fold_aucs]})
        print(f'  [{i+1:2d}/{n_iter}] mean_auc={mean_auc:.4f} '
              f'folds=[{", ".join(f"{a:.4f}" for a in fold_aucs)}] '
              f'| leaves={params.get("num_leaves")}, lr={params.get("learning_rate")}, '
              f'n_est={params.get("n_estimators")}, spw={params.get("scale_pos_weight")}')

        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_params = params

    print(f'\nBest expanding-window CV AUC: {best_mean_auc:.4f}')
    print(f'Best params: {best_params}')

    print('Refitting best config on full training set...')
    best_model = build_lgbm_model(best_params)
    best_model.fit(X_train_df, y_train, categorical_feature=cat_cols)
    print('Done.')

    results_df = pd.DataFrame(records).sort_values('mean_cv_auc', ascending=False)
    return best_model, results_df

In [ ]:
def tune_nn(X_train, y_train, X_val, y_val, param_grid, epochs=50, batch_size=2048):
    """Grid-search over NN architectures with temporal validation.

    Uses the provided temporal train/val split (instead of random splitting)
    with BCEWithLogitsLoss (pos_weight for class imbalance), cosine
    annealing LR schedule, and early stopping (patience=5 on val AUC).

    Parameters
    ----------
    X_train    : np.ndarray — training features (e.g. Jan-Oct)
    y_train    : np.ndarray — training labels
    X_val      : np.ndarray — temporal validation features (e.g. Nov)
    y_val      : np.ndarray — temporal validation labels
    param_grid : dict
        Keys: 'hidden_layers', 'learning_rate', 'activation'.
    epochs     : int — max training epochs per configuration
    batch_size : int

    Returns
    -------
    best_model : FPDNet (CPU, eval mode)
    results_df : DataFrame — val_auc for every tested configuration
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    tr_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    val_X = torch.tensor(X_val, dtype=torch.float32).to(device)

    pos_weight = torch.tensor([(1 - y_train.mean()) / y_train.mean()]).to(device)
    print(f'NN training: pos_weight={pos_weight.item():.2f}, device={device}, '
          f'max_epochs={epochs}, batch_size={batch_size}')

    input_dim = X_train.shape[1]
    combos = list(product(
        param_grid.get('hidden_layers', [[64, 32]]),
        param_grid.get('learning_rate', [1e-3]),
        param_grid.get('activation', ['relu']),
    ))
    print(f'Testing {len(combos)} configurations...\n')

    records = []
    best_auc = -1
    best_state = None
    best_cfg = None

    for hl, lr, act in tqdm(combos, desc='NN grid search'):
        model = build_nn_model(input_dim, hl, act).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

        best_epoch_auc = -1
        best_epoch_state = None
        patience_counter = 0
        patience = 5
        stopped_epoch = epochs

        for epoch in range(epochs):
            model.train()
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                preds = model.predict_proba(val_X).cpu().numpy()
            epoch_auc = roc_auc_score(y_val, preds)

            if epoch_auc > best_epoch_auc:
                best_epoch_auc = epoch_auc
                best_epoch_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    stopped_epoch = epoch + 1
                    break

        auc = best_epoch_auc
        records.append(dict(
            hidden_layers=str(hl), lr=lr, activation=act,
            val_auc=auc, stopped_epoch=stopped_epoch,
        ))
        print(f'  {hl} lr={lr} {act}: AUC={auc:.4f} (stopped @ epoch {stopped_epoch})')

        if auc > best_auc:
            best_auc = auc
            best_state = best_epoch_state
            best_cfg = (hl, lr, act)

    results_df = pd.DataFrame(records).sort_values('val_auc', ascending=False)
    print(f'\nBest NN val AUC: {best_auc:.4f}')
    print(f'Config: hidden_layers={best_cfg[0]}, lr={best_cfg[1]}, act={best_cfg[2]}')

    best_model = build_nn_model(input_dim, best_cfg[0], best_cfg[2])
    best_model.load_state_dict(best_state)
    best_model.eval()
    return best_model, results_df

## Execution — Run Full Pipeline

Toggle features in `selected_features`, then run the cells below sequentially.

In [ ]:
# ── XGBoost / NN Feature selection ───────────────────────────────────────────
selected_features = [
    # Raw numeric — dollar amounts and KYC verification scores
    'FINANCE_AMOUNT', 'PURCHASE_AMOUNT', 'TOTAL_DUE', 'DOWN_PAYMENT_AMOUNT',
    'FACE_RECOGNITION_SCORE', 'IDVALIDATION_OVERALL_SCORE',
    'LIVENESS_SCORE', 'OVERALL_SCORE',
    # Engineered numeric — ratios and time signals
    'down_payment_ratio', 'finance_ratio',
    'tx_hour', 'tx_dayofweek', 'tx_month',
    'merchant_tenure_days',
    # v4: days between transaction and first payment due date
    'days_to_first_due',
    # Smoothed entity-level FPD rates (computed from train fold only)
    'merchantid_fpd_rate', 'clerk_id_fpd_rate', 'adminid_fpd_rate',
    'state_fpd_rate',
    # v4: CITY and MODEL smoothed target encoding
    'city_fpd_rate', 'model_fpd_rate',
    # Entity order volume (temporally stable maturity signal)
    'merchantid_order_count', 'clerk_id_order_count', 'adminid_order_count',
    # Categorical — one-hot encoded by prep_data
    'COUNTRY', 'LOCK_PRODUCT', 'MANUFACTURER',
    'LOCK_NAME', 'CURRENCY',
]

# ── Temporal validation split (XGBoost / NN) ─────────────────────────────────
import time
_start = time.time()
print('[PROGRESS] Step 1/5: Preparing XGBoost / NN data (OHE + scaling)...')
X_tr, X_val, X_test, y_tr, y_val, scaler = prep_data(
    orders, payments, selected_features, val_month=11,
)
print(f'Train (Jan-Oct): {X_tr.shape}  Val (Nov): {X_val.shape}  '
      f'Test (Dec): {X_test.shape}')
print(f'Train FPD rate: {y_tr.mean():.4f}  Val FPD rate: {y_val.mean():.4f}')
print(f'[PROGRESS] Data prep done in {time.time() - _start:.1f}s')

In [ ]:
# ── XGBoost hyperparameter search ────────────────────────────────────────────
print('[PROGRESS] Step 2/5: XGBoost tuning starting...')
xgb_param_grid = {
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 500, 800],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'scale_pos_weight': [1, 5, 9],
}

# 5-fold CV × 10 random combos on the Jan-Oct training set.
best_xgb, xgb_results = tune_xgb(X_tr, y_tr, xgb_param_grid, cv_folds=5, n_iter=10)

# Evaluate on the held-out November set (our best proxy for Dec performance).
xgb_val_probs = best_xgb.predict_proba(X_val)[:, 1]
xgb_nov_auc = roc_auc_score(y_val, xgb_val_probs)
plot_roc_curve(y_val, xgb_val_probs, label='XGBoost (Nov holdout)')
print(f'\nNovember held-out AUC: {xgb_nov_auc:.4f}')

del best_xgb
gc.collect()
print('[PROGRESS] Step 2/5: XGBoost done.')

In [ ]:
# ── LightGBM: data prep + hyperparameter search + evaluation ─────────────────
print('[PROGRESS] Step 3/5: LightGBM tuning starting...')
lgbm_numeric_features = [
    'FINANCE_AMOUNT', 'PURCHASE_AMOUNT', 'TOTAL_DUE', 'DOWN_PAYMENT_AMOUNT',
    'FACE_RECOGNITION_SCORE', 'IDVALIDATION_OVERALL_SCORE',
    'LIVENESS_SCORE', 'OVERALL_SCORE',
    'down_payment_ratio', 'finance_ratio',
    'tx_hour', 'tx_dayofweek', 'tx_month',
    'merchant_tenure_days', 'days_to_first_due',
    'merchantid_fpd_rate', 'clerk_id_fpd_rate', 'adminid_fpd_rate',
    'state_fpd_rate', 'city_fpd_rate', 'model_fpd_rate',
    'merchantid_order_count', 'clerk_id_order_count', 'adminid_order_count',
]

lgbm_cat_features = [
    'CITY', 'MODEL', 'MERCHANTID', 'CLERK_ID', 'ADMINID',
    'COUNTRY', 'LOCK_PRODUCT', 'MANUFACTURER', 'LOCK_NAME', 'CURRENCY',
]

lgbm_features = lgbm_numeric_features + lgbm_cat_features

print('Preparing LightGBM data (native categoricals, no OHE)...')
lgbm_result = prep_data_lgbm(orders, payments, lgbm_features,
                              lgbm_cat_features, val_month=11)
lgbm_X_tr, lgbm_X_val, lgbm_X_test = lgbm_result[0], lgbm_result[1], lgbm_result[2]
lgbm_y_tr, lgbm_y_val = lgbm_result[3], lgbm_result[4]
lgbm_cat_cols, lgbm_train_months = lgbm_result[5], lgbm_result[6]

print(f'LightGBM Train: {lgbm_X_tr.shape}  Val: {lgbm_X_val.shape}  '
      f'Test: {lgbm_X_test.shape}')

assert np.array_equal(lgbm_y_tr, y_tr), 'LightGBM/XGBoost training labels mismatch!'
assert np.array_equal(lgbm_y_val, y_val), 'LightGBM/XGBoost validation labels mismatch!'
print('Label alignment with XGBoost split: OK')

# ── LightGBM hyperparameter search ──────────────────────────────────────────
lgbm_param_grid = {
    'num_leaves': [15, 31, 63],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [200, 500],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'min_child_samples': [20, 50, 100],
    'scale_pos_weight': [3, 5, 7],
}

best_lgbm, lgbm_results = tune_lgbm(
    lgbm_X_tr, lgbm_y_tr, lgbm_train_months, lgbm_param_grid,
    cat_cols=lgbm_cat_cols, n_iter=10,
)

# ── Evaluate on November holdout ─────────────────────────────────────────────
lgbm_val_probs = best_lgbm.predict_proba(lgbm_X_val)[:, 1]
lgbm_nov_auc = roc_auc_score(lgbm_y_val, lgbm_val_probs)
plot_roc_curve(lgbm_y_val, lgbm_val_probs, label='LightGBM (Nov holdout)')
print(f'\nLightGBM November held-out AUC: {lgbm_nov_auc:.4f}')

print('\n=== LightGBM Top-5 Configurations ===')
print(lgbm_results.head())

del best_lgbm, lgbm_results
gc.collect()
print('[PROGRESS] Step 3/5: LightGBM done.')

In [ ]:
# ── Neural Network hyperparameter search ─────────────────────────────────────
print('[PROGRESS] Step 4/5: Neural Network tuning starting...')
nn_param_grid = {
    'hidden_layers': [[256, 128], [128, 64, 32]],
    'learning_rate': [1e-3, 5e-4],
    'activation': ['relu'],
}

# Temporal validation: X_tr (Jan-Oct) trains, X_val (Nov) evaluates
best_nn, nn_results = tune_nn(X_tr, y_tr, X_val, y_val, nn_param_grid,
                              epochs=50, batch_size=1024)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_nn.to(device)
with torch.no_grad():
    val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    nn_val_probs = best_nn.predict_proba(val_tensor).cpu().numpy()
nn_nov_auc = roc_auc_score(y_val, nn_val_probs)
plot_roc_curve(y_val, nn_val_probs, label='Neural Network (Nov holdout)')
print(f'\nNN November held-out AUC: {nn_nov_auc:.4f}')

# ── Side-by-side summary ────────────────────────────────────────────────────
print('\n=== XGBoost Top-5 Configurations ===')
print(xgb_results.head())
print(f'XGBoost Nov AUC: {xgb_nov_auc:.4f}')
print('\n=== Neural Network Top-5 Configurations ===')
print(nn_results.head())
print(f'NN Nov AUC: {nn_nov_auc:.4f}')

del best_nn, nn_results, xgb_results
gc.collect()
print('[PROGRESS] Step 4/5: Neural Network done.')

In [ ]:
# ── Ensemble Evaluation on November Holdout ──────────────────────────────────
print('[PROGRESS] Step 5/5: Ensemble evaluation starting...')
print('=' * 60)
print('ENSEMBLE EVALUATION — November Holdout')
print('=' * 60)

print(f'\nIndividual model AUCs:')
print(f'  XGBoost:  {xgb_nov_auc:.4f}')
print(f'  LightGBM: {lgbm_nov_auc:.4f}')
print(f'  NN:       {nn_nov_auc:.4f}')

# Simple average ensemble
avg_probs = (xgb_val_probs + lgbm_val_probs + nn_val_probs) / 3
avg_auc = roc_auc_score(y_val, avg_probs)
print(f'\nSimple average ensemble AUC: {avg_auc:.4f}')

# Weighted grid search
best_w_auc = 0
best_weights = None
weight_records = []

for w_xgb in np.arange(0.1, 0.9, 0.1):
    for w_lgbm in np.arange(0.1, 0.9 - w_xgb + 0.01, 0.1):
        w_nn = round(1.0 - w_xgb - w_lgbm, 2)
        if w_nn < 0.05:
            continue
        blend = w_xgb * xgb_val_probs + w_lgbm * lgbm_val_probs + w_nn * nn_val_probs
        auc = roc_auc_score(y_val, blend)
        weight_records.append({
            'w_xgb': round(w_xgb, 1), 'w_lgbm': round(w_lgbm, 1),
            'w_nn': round(w_nn, 1), 'auc': auc,
        })
        if auc > best_w_auc:
            best_w_auc = auc
            best_weights = (round(w_xgb, 1), round(w_lgbm, 1), round(w_nn, 1))

print(f'\nBest weighted ensemble AUC: {best_w_auc:.4f}')
print(f'Weights: XGB={best_weights[0]:.1f}, LGBM={best_weights[1]:.1f}, NN={best_weights[2]:.1f}')

# Top-5 weight combinations
weight_df = pd.DataFrame(weight_records).sort_values('auc', ascending=False)
print(f'\nTop-5 weight combinations:')
print(weight_df.head().to_string(index=False))

# Final summary
print(f'\n{"=" * 60}')
print(f'FINAL RESULTS SUMMARY')
print(f'{"=" * 60}')
print(f'{"Model":<30} {"Nov AUC":>10}')
print(f'{"-" * 42}')
print(f'{"XGBoost (solo)":<30} {xgb_nov_auc:>10.4f}')
print(f'{"LightGBM (solo)":<30} {lgbm_nov_auc:>10.4f}')
print(f'{"Neural Network (solo)":<30} {nn_nov_auc:>10.4f}')
print(f'{"Simple Average Ensemble":<30} {avg_auc:>10.4f}')
print(f'{"Best Weighted Ensemble":<30} {best_w_auc:>10.4f}')
print('[PROGRESS] Step 5/5: Ensemble done. All experiments complete.')

## Submission Generation

Retrain the winning XGBoost model on **all** of Jan-Nov (with entity FPD rates computed from all Jan-Nov), predict December, and write `submission.csv`. This uses `val_month=None` so the full labelled dataset becomes the training set.

In [ ]:
# ── Retrain on full Jan-Nov and predict December ─────────────────────────────
# val_month=None → all labelled rows become training data, entity rates
# computed from all Jan-Nov (mirrors real-world knowledge at prediction time).
X_train_full, X_test_full, y_train_full, scaler_full = prep_data(
    orders, payments, selected_features, val_month=None,
)
print(f'Full training set: {X_train_full.shape}  Test set: {X_test_full.shape}')
print(f'Full train FPD rate: {y_train_full.mean():.4f}')

# Build and fit the winning XGBoost configuration (hardcoded from tuning).
best_params = {
    'max_depth': 4,
    'learning_rate': 0.1,
    'n_estimators': 500,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 5,
}
final_model = build_xgb_model(best_params)
final_model.fit(X_train_full, y_train_full)

# Predict December probabilities
dec_probs = final_model.predict_proba(X_test_full)[:, 1]

# Build submission DataFrame — must match Test_OrderIDs exactly
dec_orders = orders[orders['FPD_15'].isna()].reset_index(drop=True)
submission = pd.DataFrame({
    'FINANCEORDERID': dec_orders['FINANCEORDERID'].astype(str),
    'FPD_15_pred': dec_probs,
})
submission = submission.sort_values('FINANCEORDERID').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

# ── Validation checklist (shared util) ───────────────────────────────────────
from lib.submission_utils import validate_submission

print('\nSubmission validation:')
ok, errors = validate_submission(submission, test_ids)
for err in errors:
    print(f'  [FAIL] {err}')
if ok:
    print('  [OK] No duplicate IDs')
    print('  [OK] All test IDs present')
    print('  [OK] Probabilities in [0, 1]')
    print('  [OK] No missing values')

print(f'\nRows: {len(submission)}  Mean pred: {submission["FPD_15_pred"].mean():.4f}')
print('submission.csv written.')
submission.head()